In [1]:
# Cell 1：探索 genres 数据
import pandas as pd
import numpy as np

movies = pd.read_csv("../data/ml-1m/movies.dat", sep="::",
    names=["movie_id","title","genres"], engine="python", encoding="latin-1")

# 1. 所有 genre 类别
all_genres = set()
for g in movies["genres"]:
    for tag in g.split("|"):
        all_genres.add(tag)

genre_list = sorted(all_genres)
print(f"共 {len(genre_list)} 个 genre：")
print(genre_list)

# 2. 每个 genre 有多少部电影
genre_counts = {}
for g in movies["genres"]:
    for tag in g.split("|"):
        genre_counts[tag] = genre_counts.get(tag, 0) + 1

genre_series = pd.Series(genre_counts).sort_values(ascending=False)
print("\n各 genre 电影数量：")
print(genre_series.to_string())

# 3. 每部电影平均几个 genre
movies["n_genres"] = movies["genres"].apply(lambda x: len(x.split("|")))
print(f"\n每部电影 genre 数量：mean={movies['n_genres'].mean():.2f}, "
      f"min={movies['n_genres'].min()}, max={movies['n_genres'].max()}")

# 4. 看几个例子
print("\n随机 5 部电影：")
print(movies[["movie_id","title","genres"]].sample(5).to_string(index=False))

共 18 个 genre：
['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

各 genre 电影数量：
Drama          1603
Comedy         1200
Action          503
Thriller        492
Romance         471
Horror          343
Adventure       283
Sci-Fi          276
Children's      251
Crime           211
War             143
Documentary     127
Musical         114
Mystery         106
Animation       105
Fantasy          68
Western          68
Film-Noir        44

每部电影 genre 数量：mean=1.65, min=1, max=6

随机 5 部电影：
 movie_id                        title                    genres
       69                Friday (1995)                    Comedy
     1473              Best Men (1997) Action|Comedy|Crime|Drama
     1899 Passion in the Desert (1998)           Adventure|Drama
     2321         Pleasantville (1998)                    Comedy
     1113        Associate, The (19

In [2]:
# Cell 2：构建 genre 多热编码矩阵
import torch

# 固定 genre 顺序（后面模型要用，顺序不能变）
genre_list = ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 
              'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 
              'Thriller', 'War', 'Western']
genre2idx = {g: i for i, g in enumerate(genre_list)}
N_GENRES = len(genre_list)  # 18

# 构建 movie_id → 多热向量 的映射
# 注意：movie_id 不连续（1~3952，有空缺），用字典存
movie_genre_dict = {}
for _, row in movies.iterrows():
    vec = np.zeros(N_GENRES, dtype=np.float32)
    for tag in row["genres"].split("|"):
        if tag in genre2idx:
            vec[genre2idx[tag]] = 1.0
    movie_genre_dict[row["movie_id"]] = vec

# 转成矩阵，行 = movie_id 在 movies 表中的顺序
# 同时建立 movie_id → row_index 映射（后面 Dataset 里要用）
movie_ids = movies["movie_id"].values
movie2row = {mid: i for i, mid in enumerate(movie_ids)}

genre_matrix = np.stack([movie_genre_dict[mid] for mid in movie_ids])  # [3883, 18]
genre_tensor = torch.tensor(genre_matrix, dtype=torch.float32)

print(f"genre_tensor shape: {genre_tensor.shape}")  # 期望 [3883, 18]
print(f"movie_ids[:5]: {movie_ids[:5]}")
print(f"movie2row 前5项: { {k: movie2row[k] for k in movie_ids[:5]} }")

# 验证一部电影
test_id = 1  # Toy Story
test_vec = genre_tensor[movie2row[test_id]]
hit_genres = [genre_list[i] for i in range(N_GENRES) if test_vec[i] > 0]
print(f"\nmovie_id=1 ({movies[movies.movie_id==1]['title'].values[0]})")
print(f"  genres 原始: {movies[movies.movie_id==1]['genres'].values[0]}")
print(f"  genres 向量命中: {hit_genres}")

# 顺便检查：有没有全零向量（即没有任何 genre 的电影）
zero_rows = (genre_tensor.sum(dim=1) == 0).sum().item()
print(f"\n全零向量（无 genre）的电影数：{zero_rows}")

genre_tensor shape: torch.Size([3883, 18])
movie_ids[:5]: [1 2 3 4 5]
movie2row 前5项: {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2, np.int64(4): 3, np.int64(5): 4}

movie_id=1 (Toy Story (1995))
  genres 原始: Animation|Children's|Comedy
  genres 向量命中: ['Animation', "Children's", 'Comedy']

全零向量（无 genre）的电影数：0


In [3]:
# Cell 3：Content-Aware NCF 模型定义
import torch
import torch.nn as nn

class ContentNCF(nn.Module):
    def __init__(self, n_users, n_items, genre_tensor, emb_dim=64, mlp_dims=[256,128,64]):
        super().__init__()
        
        # genre 特征矩阵：固定，不参与训练
        # register_buffer：保存/加载权重时自动处理，推理时自动跟模型走同一个 device
        self.register_buffer("genre_tensor", genre_tensor)  # [n_items_in_dict, 18]
        
        n_genres = genre_tensor.shape[1]  # 18
        
        # MF 路径 embedding（和 Week 8 一样，分开）
        self.mf_user_emb = nn.Embedding(n_users + 1, emb_dim)
        self.mf_item_emb = nn.Embedding(n_items + 1, emb_dim)
        
        # MLP 路径 embedding
        self.mlp_user_emb = nn.Embedding(n_users + 1, emb_dim)
        self.mlp_item_emb = nn.Embedding(n_items + 1, emb_dim)
        
        # MLP 路径：输入维度 = emb_dim(user) + emb_dim(item) + n_genres
        mlp_input_dim = emb_dim + emb_dim + n_genres  # 64 + 64 + 18 = 146
        
        layers = []
        in_dim = mlp_input_dim
        for out_dim in mlp_dims:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        
        # 最终输出：MF 输出(64) + MLP 最后一层(64) → 1
        self.output_layer = nn.Linear(emb_dim + mlp_dims[-1], 1)
        
        # 初始化
        for emb in [self.mf_user_emb, self.mf_item_emb,
                    self.mlp_user_emb, self.mlp_item_emb]:
            nn.init.normal_(emb.weight, std=0.01)
    
    def forward(self, user_ids, item_row_indices):
        # item_row_indices：movie_id 在 genre_tensor 中的行号（不是 movie_id 本身）
        
        # MF 路径
        mf_u = self.mf_user_emb(user_ids)           # [B, 64]
        mf_i = self.mf_item_emb(item_row_indices)    # [B, 64]
        mf_out = mf_u * mf_i                         # [B, 64] 逐元素乘
        
        # MLP 路径
        mlp_u = self.mlp_user_emb(user_ids)          # [B, 64]
        mlp_i = self.mlp_item_emb(item_row_indices)  # [B, 64]
        genre_vec = self.genre_tensor[item_row_indices]  # [B, 18]
        mlp_in = torch.cat([mlp_u, mlp_i, genre_vec], dim=1)  # [B, 146]
        mlp_out = self.mlp(mlp_in)                   # [B, 64]
        
        # 合并输出
        out = self.output_layer(torch.cat([mf_out, mlp_out], dim=1))  # [B, 1]
        return torch.sigmoid(out).squeeze(1)  # [B]

# 快速验证：forward 能跑通，shape 正确
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {device}")

# 临时用假参数测试
_model = ContentNCF(n_users=6040, n_items=3883, genre_tensor=genre_tensor).to(device)
_u = torch.tensor([0, 1, 2], dtype=torch.long).to(device)
_i = torch.tensor([0, 1, 2], dtype=torch.long).to(device)
_out = _model(_u, _i)
print(f"output shape: {_out.shape}")   # 期望 torch.Size([3])
print(f"output values: {_out}")        # 期望在 (0, 1) 之间

# 打印参数量
total = sum(p.numel() for p in _model.parameters())
print(f"总参数量：{total:,}")

device: mps
output shape: torch.Size([3])
output values: tensor([0.4824, 0.4827, 0.4836], device='mps:0', grad_fn=<SqueezeBackward1>)
总参数量：1,349,313


In [4]:
# Cell 4：Dataset（含负采样 + movie_id→row_index 映射）
from torch.utils.data import Dataset, DataLoader

ratings = pd.read_csv("../data/ml-1m/ratings.dat", sep="::",
    names=["user_id","movie_id","rating","timestamp"], engine="python")

# --- 按时间戳切分（和 Week 8 一致，防止 data leakage）---
ratings_sorted = ratings.sort_values("timestamp")
n = len(ratings_sorted)
train_df = ratings_sorted.iloc[:int(n * 0.8)].copy()
val_df   = ratings_sorted.iloc[int(n * 0.8):].copy()

# --- ID 重映射（连续化） ---
all_user_ids  = ratings["user_id"].unique()
all_movie_ids = ratings["movie_id"].unique()

user2idx = {u: i for i, u in enumerate(sorted(all_user_ids))}
item2idx = {m: i for i, m in enumerate(sorted(all_movie_ids))}  # movie_id → 连续idx

# movie_id → genre_tensor 行号（上一个 cell 已建好 movie2row）
# 注意：movie2row 是 movies 表里的顺序，item2idx 是评分表里的顺序，两者不同
# 模型 Embedding 用 item2idx，genre_tensor 查询用 movie2row

n_users = len(user2idx)
n_items = len(item2idx)
print(f"n_users={n_users}, n_items={n_items}")

# 用户交互过的 movie_id 集合（用于负采样排除）
user_pos_movies = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
all_movie_id_list = list(item2idx.keys())  # 有评分的电影列表

class ContentNCFDataset(Dataset):
    def __init__(self, df, user2idx, item2idx, movie2row, 
                 user_pos_movies, all_movie_id_list, neg_ratio=4):
        self.user2idx = user2idx
        self.item2idx = item2idx
        self.movie2row = movie2row
        self.neg_ratio = neg_ratio
        self.user_pos_movies = user_pos_movies
        self.all_movie_id_list = all_movie_id_list
        
        # 正样本
        self.pos_pairs = list(zip(df["user_id"].values, df["movie_id"].values))
    
    def __len__(self):
        return len(self.pos_pairs) * (1 + self.neg_ratio)
    
    def __getitem__(self, idx):
        pos_idx = idx // (1 + self.neg_ratio)
        is_neg  = (idx %  (1 + self.neg_ratio)) != 0
        
        user_id, movie_id = self.pos_pairs[pos_idx]
        
        if is_neg:
            # 负采样：随机取一个用户没交互过的电影
            while True:
                neg_movie_id = self.all_movie_id_list[
                    np.random.randint(len(self.all_movie_id_list))]
                if neg_movie_id not in self.user_pos_movies.get(user_id, set()):
                    movie_id = neg_movie_id
                    break
            label = 0.0
        else:
            label = 1.0
        
        return (
            torch.tensor(self.user2idx[user_id],   dtype=torch.long),
            torch.tensor(self.item2idx[movie_id],  dtype=torch.long),   # embedding 用
            torch.tensor(self.movie2row[movie_id], dtype=torch.long),   # genre 查询用
            torch.tensor(label, dtype=torch.float32)
        )

train_dataset = ContentNCFDataset(train_df, user2idx, item2idx, movie2row,
                                   user_pos_movies, all_movie_id_list, neg_ratio=4)
val_dataset   = ContentNCFDataset(val_df,   user2idx, item2idx, movie2row,
                                   user_pos_movies, all_movie_id_list, neg_ratio=4)

train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=2048, shuffle=False, num_workers=0)

print(f"train 正样本: {len(train_df):,}  →  Dataset 总量: {len(train_dataset):,}")
print(f"val   正样本: {len(val_df):,}  →  Dataset 总量: {len(val_dataset):,}")

# 验证一个 batch 的 shape
u, i_idx, i_row, lbl = next(iter(train_loader))
print(f"\nbatch shapes — user:{u.shape}, item_idx:{i_idx.shape}, "
      f"item_row:{i_row.shape}, label:{lbl.shape}")
print(f"label 分布 — pos:{lbl.sum().item():.0f}, neg:{(lbl==0).sum().item():.0f}")

n_users=6040, n_items=3706
train 正样本: 800,167  →  Dataset 总量: 4,000,835
val   正样本: 200,042  →  Dataset 总量: 1,000,210

batch shapes — user:torch.Size([2048]), item_idx:torch.Size([2048]), item_row:torch.Size([2048]), label:torch.Size([2048])
label 分布 — pos:419, neg:1629


In [7]:
# Cell 5：训练
import torch.optim as optim

# 重新实例化模型（用真实的 n_users/n_items）
model = ContentNCF(
    n_users=n_users,
    n_items=n_items,
    genre_tensor=genre_tensor
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.BCELoss()

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_samples = 0.0, 0
    for u, i_idx, i_row, lbl in loader:
        u, i_idx, i_row, lbl = u.to(device), i_idx.to(device), i_row.to(device), lbl.to(device)
        optimizer.zero_grad()
        pred = model(u, i_row)   # ← 注意：embedding 和 genre 都用 i_row
        loss = criterion(pred, lbl)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(lbl)
        total_samples += len(lbl)
    return total_loss / total_samples

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    with torch.no_grad():
        for u, i_idx, i_row, lbl in loader:
            u, i_idx, i_row, lbl = u.to(device), i_idx.to(device), i_row.to(device), lbl.to(device)
            pred = model(u, i_row)
            loss = criterion(pred, lbl)
            total_loss += loss.item() * len(lbl)
            total_samples += len(lbl)
    return total_loss / total_samples

# 训练 10 epoch，观察 loss 曲线
N_EPOCHS = 10
best_val_loss = float("inf")
best_epoch = -1

print(f"{'Epoch':>5} {'Train Loss':>12} {'Val Loss':>10}")
print("-" * 32)

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss   = eval_epoch(model, val_loader, criterion, device)
    
    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), "../data/content_ncf_best.pth")
        marker = " ✓"
    
    print(f"{epoch:>5} {train_loss:>12.6f} {val_loss:>10.6f}{marker}")

print(f"\n最佳 val loss: {best_val_loss:.6f}（epoch {best_epoch}）")

Epoch   Train Loss   Val Loss
--------------------------------
    1     0.336834   0.420282 ✓
    2     0.287257   0.434513
    3     0.273610   0.443341
    4     0.266079   0.446256
    5     0.260644   0.458947
    6     0.256751   0.462055
    7     0.253484   0.470335
    8     0.250311   0.473148
    9     0.247976   0.473632
   10     0.245500   0.480820

最佳 val loss: 0.420282（epoch 1）


In [8]:
# Cell 6：诊断 overfitting 原因
# 怀疑方向：val 集里有大量"冷用户"（训练集交互很少），模型没见过

# 1. 每个用户在 train/val 里各有多少条评分
train_user_counts = train_df.groupby("user_id").size()
val_user_counts   = val_df.groupby("user_id").size()

print("=== 用户在 train 集的评分数分布 ===")
print(train_user_counts.describe().round(1))

print("\n=== 用户在 val 集的评分数分布 ===")
print(val_user_counts.describe().round(1))

# 2. val 集里有多少用户在 train 里评分数 < 5（几乎没见过）
val_users = val_df["user_id"].unique()
cold_users = [u for u in val_users if train_user_counts.get(u, 0) < 5]
print(f"\nval 集用户总数: {len(val_users)}")
print(f"其中 train 评分 < 5 的用户: {len(cold_users)} ({len(cold_users)/len(val_users)*100:.1f}%)")

# 3. val 集里有多少电影在 train 里出现次数 < 5
train_movie_counts = train_df.groupby("movie_id").size()
val_movies = val_df["movie_id"].unique()
cold_movies = [m for m in val_movies if train_movie_counts.get(m, 0) < 5]
print(f"\nval 集电影总数: {len(val_movies)}")
print(f"其中 train 出现 < 5 次的电影: {len(cold_movies)} ({len(cold_movies)/len(val_movies)*100:.1f}%)")

# 4. 按时间切分的副作用：val 集是"更新"的评分
# 看看 train/val 的时间戳范围
import datetime
print(f"\ntrain 时间范围: "
      f"{datetime.datetime.fromtimestamp(train_df['timestamp'].min()).strftime('%Y-%m')} ~ "
      f"{datetime.datetime.fromtimestamp(train_df['timestamp'].max()).strftime('%Y-%m')}")
print(f"val   时间范围: "
      f"{datetime.datetime.fromtimestamp(val_df['timestamp'].min()).strftime('%Y-%m')} ~ "
      f"{datetime.datetime.fromtimestamp(val_df['timestamp'].max()).strftime('%Y-%m')}")

=== 用户在 train 集的评分数分布 ===
count    5400.0
mean      148.2
std       170.6
min         2.0
25%        41.0
50%        86.0
75%       187.0
max      1849.0
dtype: float64

=== 用户在 val 集的评分数分布 ===
count    1783.0
mean      112.2
std       149.1
min         1.0
25%        24.0
50%        57.0
75%       137.5
max      1226.0
dtype: float64

val 集用户总数: 1783
其中 train 评分 < 5 的用户: 648 (36.3%)

val 集电影总数: 3511
其中 train 出现 < 5 次的电影: 200 (5.7%)

train 时间范围: 2000-04 ~ 2000-12
val   时间范围: 2000-12 ~ 2003-02


In [9]:
# Cell 7：HR@10 / NDCG@10 评估（与 Week 8 对比）
model.load_state_dict(torch.load("../data/content_ncf_best.pth", map_location=device))
model.eval()

def evaluate_hr_ndcg(model, val_df, train_df, user2idx, item2idx, 
                     movie2row, device, k=10, n_neg=99, n_users_eval=500):
    """
    评估协议：每个用户取最后1条正样本 + 99个负样本，看正样本是否进 Top-K
    与 Week 8 保持一致
    """
    # val 集每个用户最后一条评分作为正样本
    val_last = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
    
    # 只评估在 item2idx 里的电影（有评分的 3706 部）
    valid_movie_ids = list(item2idx.keys())
    train_pos = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
    
    hrs, ndcgs = [], []
    eval_users = val_last[val_last["user_id"].isin(user2idx)].sample(
        min(n_users_eval, len(val_last)), random_state=42)
    
    with torch.no_grad():
        for _, row in eval_users.iterrows():
            uid = row["user_id"]
            pos_mid = row["movie_id"]
            if pos_mid not in item2idx:
                continue
            
            # 采 99 个负样本
            seen = train_pos.get(uid, set()) | {pos_mid}
            negs = []
            candidates = [m for m in valid_movie_ids if m not in seen]
            neg_sample = np.random.choice(candidates, 
                                          size=min(n_neg, len(candidates)), 
                                          replace=False).tolist()
            
            all_mids = [pos_mid] + neg_sample  # 正样本放第一位
            
            u_tensor = torch.tensor(
                [user2idx[uid]] * len(all_mids), dtype=torch.long).to(device)
            row_tensor = torch.tensor(
                [movie2row[m] for m in all_mids], dtype=torch.long).to(device)
            
            scores = model(u_tensor, row_tensor).cpu().numpy()
            
            # 正样本的排名
            rank = (scores > scores[0]).sum() + 1  # 从1开始
            
            hrs.append(1.0 if rank <= k else 0.0)
            ndcgs.append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)
    
    return np.mean(hrs), np.mean(ndcgs), len(hrs)

hr, ndcg, n_eval = evaluate_hr_ndcg(
    model, val_df, train_df, user2idx, item2idx, movie2row, device)

print(f"评估用户数: {n_eval}")
print(f"Content NCF  HR@10:   {hr:.4f}")
print(f"Content NCF  NDCG@10: {ndcg:.4f}")
print(f"\nWeek 8 NCF   HR@10:   0.5490  (baseline)")
print(f"差值: {hr - 0.549:+.4f}")

评估用户数: 500
Content NCF  HR@10:   0.3820
Content NCF  NDCG@10: 0.2033

Week 8 NCF   HR@10:   0.5490  (baseline)
差值: -0.1670


In [10]:
# Cell 8：降低 lr + scheduler，缓解 overfit
model2 = ContentNCF(
    n_users=n_users,
    n_items=n_items,
    genre_tensor=genre_tensor
).to(device)

optimizer2 = optim.Adam(model2.parameters(), lr=3e-4, weight_decay=1e-4)

# CosineAnnealingLR：lr 从 3e-4 缓慢衰减到 1e-5，避免后期步子太大
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=15, eta_min=1e-5)

N_EPOCHS2 = 15
best_val_loss2 = float("inf")
best_epoch2 = -1

print(f"{'Epoch':>5} {'Train Loss':>12} {'Val Loss':>10} {'LR':>10}")
print("-" * 45)

for epoch in range(1, N_EPOCHS2 + 1):
    train_loss = train_epoch(model2, train_loader, optimizer2, criterion, device)
    val_loss   = eval_epoch(model2, val_loader,   criterion, device)
    scheduler.step()
    
    current_lr = scheduler.get_last_lr()[0]
    marker = ""
    if val_loss < best_val_loss2:
        best_val_loss2 = val_loss
        best_epoch2 = epoch
        torch.save(model2.state_dict(), "../data/content_ncf_v2.pth")
        marker = " ✓"
    
    print(f"{epoch:>5} {train_loss:>12.6f} {val_loss:>10.6f} {current_lr:>10.2e}{marker}")

print(f"\n最佳 val loss: {best_val_loss2:.6f}（epoch {best_epoch2}）")

Epoch   Train Loss   Val Loss         LR
---------------------------------------------
    1     0.362809   0.405906   2.97e-04 ✓
    2     0.319906   0.412216   2.87e-04
    3     0.296993   0.420777   2.72e-04
    4     0.283068   0.425858   2.52e-04
    5     0.274837   0.427965   2.27e-04
    6     0.269630   0.432101   2.00e-04
    7     0.264993   0.435029   1.70e-04
    8     0.260882   0.438107   1.40e-04
    9     0.257104   0.442695   1.10e-04
   10     0.253281   0.449507   8.25e-05
   11     0.250004   0.453162   5.80e-05
   12     0.246571   0.457063   3.77e-05
   13     0.244444   0.462304   2.25e-05
   14     0.242574   0.464747   1.32e-05
   15     0.241281   0.467148   1.00e-05

最佳 val loss: 0.405906（epoch 1）


In [11]:
# Cell 9：Two-Tower 模型
# User Tower：user_id embedding + 人口属性（gender/age/occupation）
# Item Tower：item_id embedding + genre 多热向量
# 训练目标：正样本 user/item 向量内积高，负样本内积低（同 NCF 的 BCE）

import torch
import torch.nn as nn

# 先准备 user 属性特征
users = pd.read_csv("../data/ml-1m/users.dat", sep="::",
    names=["user_id","gender","age","occupation","zip"], engine="python")

# gender: M/F → 0/1
users["gender_idx"] = (users["gender"] == "M").astype(int)

# age: 原始是分组值(1,18,25,35,45,50,56) → 重映射到 0~6
age_vals = sorted(users["age"].unique())
age2idx = {a: i for i, a in enumerate(age_vals)}
users["age_idx"] = users["age"].map(age2idx)

# occupation: 0~20，直接用
# zip: 忽略（太稀疏，信号弱）

user_attr = users.set_index("user_id")[["gender_idx","age_idx","occupation"]].to_dict("index")
# user_attr[user_id] = {"gender_idx":0, "age_idx":3, "occupation":7}

N_GENDERS     = 2
N_AGES        = len(age_vals)   # 7
N_OCCUPATIONS = 21

print(f"age 分组值: {age_vals}")
print(f"N_AGES={N_AGES}, N_OCCUPATIONS={N_OCCUPATIONS}")
print(f"user_attr 样例 (user_id=1): {user_attr[1]}")

class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, genre_tensor,
                 emb_dim=64, tower_dim=64):
        super().__init__()
        self.register_buffer("genre_tensor", genre_tensor)
        n_genres = genre_tensor.shape[1]  # 18

        # ---------- User Tower ----------
        self.user_id_emb    = nn.Embedding(n_users + 1, emb_dim)
        self.gender_emb     = nn.Embedding(N_GENDERS, 8)
        self.age_emb        = nn.Embedding(N_AGES, 8)
        self.occupation_emb = nn.Embedding(N_OCCUPATIONS, 8)

        # user tower MLP：emb_dim + 8+8+8 → tower_dim
        user_in = emb_dim + 8 + 8 + 8   # 88
        self.user_tower = nn.Sequential(
            nn.Linear(user_in, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, tower_dim)
        )

        # ---------- Item Tower ----------
        self.item_id_emb = nn.Embedding(n_items + 1, emb_dim)

        # item tower MLP：emb_dim + n_genres → tower_dim
        item_in = emb_dim + n_genres     # 82
        self.item_tower = nn.Sequential(
            nn.Linear(item_in, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, tower_dim)
        )

        # 初始化
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, user_ids, item_row_indices,
                gender_ids, age_ids, occupation_ids):

        # User Tower
        u_emb  = self.user_id_emb(user_ids)           # [B, 64]
        g_emb  = self.gender_emb(gender_ids)           # [B, 8]
        a_emb  = self.age_emb(age_ids)                 # [B, 8]
        o_emb  = self.occupation_emb(occupation_ids)   # [B, 8]
        u_vec  = self.user_tower(
            torch.cat([u_emb, g_emb, a_emb, o_emb], dim=1))   # [B, 64]

        # Item Tower
        i_emb  = self.item_id_emb(item_row_indices)    # [B, 64]
        g_vec  = self.genre_tensor[item_row_indices]   # [B, 18]
        i_vec  = self.item_tower(
            torch.cat([i_emb, g_vec], dim=1))          # [B, 64]

        # 内积 → sigmoid
        score = (u_vec * i_vec).sum(dim=1)             # [B]
        return torch.sigmoid(score)

# 快速验证
_model2 = TwoTower(n_users=n_users, n_items=n_items,
                   genre_tensor=genre_tensor).to(device)
_u  = torch.tensor([0,1,2], dtype=torch.long).to(device)
_i  = torch.tensor([0,1,2], dtype=torch.long).to(device)
_g  = torch.tensor([0,1,0], dtype=torch.long).to(device)
_a  = torch.tensor([2,3,1], dtype=torch.long).to(device)
_o  = torch.tensor([7,0,4], dtype=torch.long).to(device)
_out = _model2(_u, _i, _g, _a, _o)
print(f"\noutput shape: {_out.shape}")
print(f"output values: {_out}")
total = sum(p.numel() for p in _model2.parameters())
print(f"总参数量：{total:,}")

age 分组值: [np.int64(1), np.int64(18), np.int64(25), np.int64(35), np.int64(45), np.int64(50), np.int64(56)]
N_AGES=7, N_OCCUPATIONS=21
user_attr 样例 (user_id=1): {'gender_idx': 0, 'age_idx': 0, 'occupation': 10}

output shape: torch.Size([3])
output values: tensor([0.4965, 0.4958, 0.4941], device='mps:0', grad_fn=<SigmoidBackward0>)
总参数量：662,640


In [12]:
# Cell 10：精简 Two-Tower（gender/age 直接作为数值特征）
class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, genre_tensor,
                 emb_dim=64, tower_dim=64):
        super().__init__()
        self.register_buffer("genre_tensor", genre_tensor)
        n_genres = genre_tensor.shape[1]  # 18

        # age 归一化：把原始值映射到 [0,1]
        age_raw = torch.tensor([1,18,25,35,45,50,56], dtype=torch.float32)
        self.register_buffer("age_norm_min", age_raw.min())
        self.register_buffer("age_norm_range", age_raw.max() - age_raw.min())

        # ---------- User Tower ----------
        self.user_id_emb    = nn.Embedding(n_users + 1, emb_dim)
        self.occupation_emb = nn.Embedding(N_OCCUPATIONS, 16)

        # 输入：emb_dim + 16(occupation) + 1(gender) + 1(age) = 82
        user_in = emb_dim + 16 + 1 + 1
        self.user_tower = nn.Sequential(
            nn.Linear(user_in, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, tower_dim)
        )

        # ---------- Item Tower ----------
        self.item_id_emb = nn.Embedding(n_items + 1, emb_dim)

        item_in = emb_dim + n_genres  # 82
        self.item_tower = nn.Sequential(
            nn.Linear(item_in, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, tower_dim)
        )

        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, user_ids, item_row_indices,
                gender, age_raw, occupation_ids):
        # gender: float tensor [B], 值为 0.0 或 1.0
        # age_raw: float tensor [B], 原始分组值（1/18/25/35/45/50/56）

        # User Tower
        age_normed = (age_raw - self.age_norm_min) / self.age_norm_range  # [B], 0~1
        u_emb = self.user_id_emb(user_ids)             # [B, 64]
        o_emb = self.occupation_emb(occupation_ids)    # [B, 16]
        u_vec = self.user_tower(torch.cat([
            u_emb, o_emb,
            gender.unsqueeze(1),                       # [B, 1]
            age_normed.unsqueeze(1)                    # [B, 1]
        ], dim=1))                                     # [B, 82] → [B, 64]

        # Item Tower
        i_emb = self.item_id_emb(item_row_indices)    # [B, 64]
        g_vec = self.genre_tensor[item_row_indices]    # [B, 18]
        i_vec = self.item_tower(
            torch.cat([i_emb, g_vec], dim=1))         # [B, 82] → [B, 64]

        score = (u_vec * i_vec).sum(dim=1)
        return torch.sigmoid(score)

# 验证
_model3 = TwoTower(n_users=n_users, n_items=n_items,
                   genre_tensor=genre_tensor).to(device)
_u   = torch.tensor([0,1,2], dtype=torch.long).to(device)
_i   = torch.tensor([0,1,2], dtype=torch.long).to(device)
_g   = torch.tensor([0.,1.,0.], dtype=torch.float32).to(device)
_a   = torch.tensor([25.,35.,18.], dtype=torch.float32).to(device)
_o   = torch.tensor([7,0,4], dtype=torch.long).to(device)
_out = _model3(_u, _i, _g, _a, _o)

print(f"output shape: {_out.shape}")
print(f"output values: {_out}")
total = sum(p.numel() for p in _model3.parameters())
print(f"总参数量：{total:,}")

# 确认 age 归一化正确
test_ages = torch.tensor([1.,18.,25.,35.,45.,50.,56.]).to(device)
normed = (test_ages - _model3.age_norm_min) / _model3.age_norm_range
print(f"\nage 归一化验证：")
for raw, norm in zip([1,18,25,35,45,50,56], normed.tolist()):
    print(f"  age {raw:2d} → {norm:.3f}")

output shape: torch.Size([3])
output values: tensor([0.4928, 0.4944, 0.4964], device='mps:0', grad_fn=<SigmoidBackward0>)
总参数量：661,968

age 归一化验证：
  age  1 → 0.000
  age 18 → 0.309
  age 25 → 0.436
  age 35 → 0.618
  age 45 → 0.800
  age 50 → 0.891
  age 56 → 1.000


In [13]:
# Cell 11：Two-Tower Dataset + 训练

class TwoTowerDataset(Dataset):
    def __init__(self, df, user2idx, movie2row,
                 user_attr, user_pos_movies, all_movie_id_list, neg_ratio=4):
        self.movie2row        = movie2row
        self.user_attr        = user_attr
        self.neg_ratio        = neg_ratio
        self.user_pos_movies  = user_pos_movies
        self.all_movie_id_list = all_movie_id_list
        self.user2idx         = user2idx

        self.pos_pairs = list(zip(df["user_id"].values, df["movie_id"].values))

    def __len__(self):
        return len(self.pos_pairs) * (1 + self.neg_ratio)

    def __getitem__(self, idx):
        pos_idx = idx // (1 + self.neg_ratio)
        is_neg  = (idx %  (1 + self.neg_ratio)) != 0

        user_id, movie_id = self.pos_pairs[pos_idx]

        if is_neg:
            while True:
                neg_mid = self.all_movie_id_list[
                    np.random.randint(len(self.all_movie_id_list))]
                if neg_mid not in self.user_pos_movies.get(user_id, set()):
                    movie_id = neg_mid
                    break
            label = 0.0
        else:
            label = 1.0

        attr = self.user_attr[user_id]
        return (
            torch.tensor(self.user2idx[user_id],   dtype=torch.long),
            torch.tensor(self.movie2row[movie_id], dtype=torch.long),
            torch.tensor(attr["gender_idx"],       dtype=torch.float32),
            torch.tensor(float(attr["age"]) if "age" in attr
                         else float(attr.get("age_idx", 0)), dtype=torch.float32),
            torch.tensor(attr["occupation"],       dtype=torch.long),
            torch.tensor(label,                    dtype=torch.float32)
        )

# user_attr 里的 key 名确认一下
print("user_attr[1] keys:", list(user_attr[1].keys()))

# 重建 user_attr，确保 age 字段是原始值
users_df = pd.read_csv("../data/ml-1m/users.dat", sep="::",
    names=["user_id","gender","age","occupation","zip"], engine="python")
users_df["gender_idx"] = (users_df["gender"] == "M").astype(int)

user_attr2 = {}
for _, row in users_df.iterrows():
    user_attr2[row["user_id"]] = {
        "gender_idx":  int(row["gender_idx"]),
        "age":         float(row["age"]),        # 原始值：1/18/25/35/45/50/56
        "occupation":  int(row["occupation"])
    }
print("user_attr2[1]:", user_attr2[1])

train_ds2 = TwoTowerDataset(train_df, user2idx, movie2row,
                             user_attr2, user_pos_movies, all_movie_id_list)
val_ds2   = TwoTowerDataset(val_df,   user2idx, movie2row,
                             user_attr2, user_pos_movies, all_movie_id_list)

train_loader2 = DataLoader(train_ds2, batch_size=2048, shuffle=True,  num_workers=0)
val_loader2   = DataLoader(val_ds2,   batch_size=2048, shuffle=False, num_workers=0)

# 验证一个 batch
u, i_row, g, a, o, lbl = next(iter(train_loader2))
print(f"\nbatch shapes: u{u.shape} i{i_row.shape} g{g.shape} a{a.shape} o{o.shape} lbl{lbl.shape}")
print(f"age 原始值样例: {a[:8].tolist()}")
print(f"gender 样例:    {g[:8].tolist()}")

# ---------- 训练 ----------
model_tt = TwoTower(n_users=n_users, n_items=n_items,
                    genre_tensor=genre_tensor).to(device)
optimizer_tt = optim.Adam(model_tt.parameters(), lr=1e-3, weight_decay=1e-5)

def train_epoch_tt(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_samples = 0.0, 0
    for u, i_row, g, a, o, lbl in loader:
        u, i_row, g, a, o, lbl = (x.to(device) for x in (u, i_row, g, a, o, lbl))
        optimizer.zero_grad()
        pred = model(u, i_row, g, a, o)
        loss = criterion(pred, lbl)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(lbl)
        total_samples += len(lbl)
    return total_loss / total_samples

def eval_epoch_tt(model, loader, criterion, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    with torch.no_grad():
        for u, i_row, g, a, o, lbl in loader:
            u, i_row, g, a, o, lbl = (x.to(device) for x in (u, i_row, g, a, o, lbl))
            pred = model(u, i_row, g, a, o)
            loss = criterion(pred, lbl)
            total_loss += loss.item() * len(lbl)
            total_samples += len(lbl)
    return total_loss / total_samples

N_EPOCHS_TT = 15
best_val_tt  = float("inf")
best_epoch_tt = -1

print(f"\n{'Epoch':>5} {'Train Loss':>12} {'Val Loss':>10}")
print("-" * 32)

for epoch in range(1, N_EPOCHS_TT + 1):
    tr_loss  = train_epoch_tt(model_tt, train_loader2, optimizer_tt, criterion, device)
    val_loss = eval_epoch_tt(model_tt, val_loader2,   criterion, device)

    marker = ""
    if val_loss < best_val_tt:
        best_val_tt   = val_loss
        best_epoch_tt = epoch
        torch.save(model_tt.state_dict(), "../data/two_tower_best.pth")
        marker = " ✓"

    print(f"{epoch:>5} {tr_loss:>12.6f} {val_loss:>10.6f}{marker}")

print(f"\n最佳 val loss: {best_val_tt:.6f}（epoch {best_epoch_tt}）")

user_attr[1] keys: ['gender_idx', 'age_idx', 'occupation']
user_attr2[1]: {'gender_idx': 0, 'age': 1.0, 'occupation': 10}

batch shapes: utorch.Size([2048]) itorch.Size([2048]) gtorch.Size([2048]) atorch.Size([2048]) otorch.Size([2048]) lbltorch.Size([2048])
age 原始值样例: [25.0, 35.0, 35.0, 35.0, 25.0, 18.0, 18.0, 45.0]
gender 样例:    [1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0]

Epoch   Train Loss   Val Loss
--------------------------------
    1     0.345941   0.398961 ✓
    2     0.309305   0.404214
    3     0.292228   0.407543
    4     0.286401   0.409389
    5     0.282069   0.410528
    6     0.278433   0.405309
    7     0.275412   0.412231
    8     0.273570   0.417182
    9     0.272164   0.411156
   10     0.270969   0.413469
   11     0.270077   0.413986
   12     0.269005   0.416821
   13     0.267722   0.417662
   14     0.266849   0.417590
   15     0.265741   0.414821

最佳 val loss: 0.398961（epoch 1）


In [14]:
# Cell 12：Two-Tower HR@10 评估

model_tt.load_state_dict(torch.load("../data/two_tower_best.pth", map_location=device))
model_tt.eval()

def evaluate_hr_ndcg_tt(model, val_df, train_df, user2idx, movie2row,
                         user_attr2, device, k=10, n_neg=99, n_users_eval=500):
    val_last  = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
    train_pos = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
    valid_mids = list(movie2row.keys())

    hrs, ndcgs = [], []
    eval_users = val_last[val_last["user_id"].isin(user2idx)].sample(
        min(n_users_eval, len(val_last)), random_state=42)

    with torch.no_grad():
        for _, row in eval_users.iterrows():
            uid    = row["user_id"]
            pos_mid = row["movie_id"]
            if pos_mid not in movie2row:
                continue

            seen  = train_pos.get(uid, set()) | {pos_mid}
            candidates = [m for m in valid_mids if m not in seen]
            neg_sample = np.random.choice(
                candidates, size=min(n_neg, len(candidates)), replace=False).tolist()
            all_mids = [pos_mid] + neg_sample

            attr = user_attr2[uid]
            B    = len(all_mids)
            u_t  = torch.tensor([user2idx[uid]] * B, dtype=torch.long).to(device)
            i_t  = torch.tensor([movie2row[m] for m in all_mids], dtype=torch.long).to(device)
            g_t  = torch.full((B,), attr["gender_idx"], dtype=torch.float32).to(device)
            a_t  = torch.full((B,), attr["age"],        dtype=torch.float32).to(device)
            o_t  = torch.full((B,), attr["occupation"], dtype=torch.long).to(device)

            scores = model(u_t, i_t, g_t, a_t, o_t).cpu().numpy()
            rank   = (scores > scores[0]).sum() + 1

            hrs.append(1.0 if rank <= k else 0.0)
            ndcgs.append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)

    return np.mean(hrs), np.mean(ndcgs), len(hrs)

hr_tt, ndcg_tt, n_eval = evaluate_hr_ndcg_tt(
    model_tt, val_df, train_df, user2idx, movie2row, user_attr2, device)

print("=" * 45)
print(f"{'模型':<20} {'HR@10':>8} {'NDCG@10':>10}")
print("-" * 45)
print(f"{'Week 8 NCF':<20} {'0.5490':>8} {'0.3250':>10}  (baseline)")
print(f"{'Content NCF':<20} {0.3820:>8.4f} {0.2033:>10.4f}")
print(f"{'Two-Tower':<20} {hr_tt:>8.4f} {ndcg_tt:>10.4f}")
print("=" * 45)

# 额外分析：对 cold / warm 用户分开看 Two-Tower 的表现
train_user_counts = train_df.groupby("user_id").size()
val_last = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
eval_sample = val_last[val_last["user_id"].isin(user2idx)].sample(
    min(500, len(val_last)), random_state=42)

cold_uids = [u for u in eval_sample["user_id"] if train_user_counts.get(u, 0) < 5]
warm_uids = [u for u in eval_sample["user_id"] if train_user_counts.get(u, 0) >= 5]
print(f"\neval 样本中 cold 用户（train<5）: {len(cold_uids)}")
print(f"eval 样本中 warm 用户（train≥5）: {len(warm_uids)}")

模型                      HR@10    NDCG@10
---------------------------------------------
Week 8 NCF             0.5490     0.3250  (baseline)
Content NCF            0.3820     0.2033
Two-Tower              0.4000     0.2092

eval 样本中 cold 用户（train<5）: 187
eval 样本中 warm 用户（train≥5）: 313


In [18]:
# Cell 13：用 Week 8 原始类定义加载 NCF
class NCF_orig(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32, mlp_dims=[256,128,64]):
        super().__init__()
        self.mf_user_emb  = nn.Embedding(n_users, emb_dim)  # 不加1
        self.mf_item_emb  = nn.Embedding(n_items, emb_dim)
        self.mlp_user_emb = nn.Embedding(n_users, emb_dim)
        self.mlp_item_emb = nn.Embedding(n_items, emb_dim)
        layers, in_dim = [], emb_dim * 2
        for out_dim in mlp_dims:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.output_layer = nn.Linear(emb_dim + mlp_dims[-1], 1)

    def forward(self, user_ids, item_ids):
        mf_out  = self.mf_user_emb(user_ids) * self.mf_item_emb(item_ids)
        mlp_in  = torch.cat([self.mlp_user_emb(user_ids),
                              self.mlp_item_emb(item_ids)], dim=1)
        mlp_out = self.mlp(mlp_in)
        out     = self.output_layer(torch.cat([mf_out, mlp_out], dim=1))
        return torch.sigmoid(out).squeeze(1)

model_ncf = NCF_orig(n_users=6040, n_items=3706, emb_dim=32).to(device)
model_ncf.load_state_dict(torch.load("../data/ncf_best_v2.pth", map_location=device))
model_ncf.eval()
print("NCF 加载成功")

# Week 8 用的是 item2idx（连续映射），不是 movie2row
print(f"item2idx 值域: [{min(item2idx.values())}, {max(item2idx.values())}]")
print(f"user2idx 值域: [{min(user2idx.values())}, {max(user2idx.values())}]")

RuntimeError: Error(s) in loading state_dict for NCF_orig:
	size mismatch for mlp.0.weight: copying a param with shape torch.Size([64, 64]) from checkpoint, the shape in current model is torch.Size([256, 64]).
	size mismatch for mlp.0.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for mlp.3.weight: copying a param with shape torch.Size([32, 64]) from checkpoint, the shape in current model is torch.Size([128, 256]).
	size mismatch for mlp.3.bias: copying a param with shape torch.Size([32]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for mlp.6.weight: copying a param with shape torch.Size([16, 32]) from checkpoint, the shape in current model is torch.Size([64, 128]).
	size mismatch for mlp.6.bias: copying a param with shape torch.Size([16]) from checkpoint, the shape in current model is torch.Size([64]).
	size mismatch for output_layer.weight: copying a param with shap

In [19]:
# 查看 checkpoint 里所有层的 shape
ckpt = torch.load("../data/ncf_best_v2.pth", map_location="cpu")
for k, v in ckpt.items():
    print(f"{k:40s} {v.shape}")

mf_user_emb.weight                       torch.Size([6040, 32])
mf_item_emb.weight                       torch.Size([3706, 32])
mlp_user_emb.weight                      torch.Size([6040, 32])
mlp_item_emb.weight                      torch.Size([3706, 32])
mlp.0.weight                             torch.Size([64, 64])
mlp.0.bias                               torch.Size([64])
mlp.3.weight                             torch.Size([32, 64])
mlp.3.bias                               torch.Size([32])
mlp.6.weight                             torch.Size([16, 32])
mlp.6.bias                               torch.Size([16])
output_layer.weight                      torch.Size([1, 48])
output_layer.bias                        torch.Size([1])


In [20]:
# 从 checkpoint shape 反推正确架构
class NCF_orig(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32, mlp_dims=[64,32,16]):
        super().__init__()
        self.mf_user_emb  = nn.Embedding(n_users, emb_dim)
        self.mf_item_emb  = nn.Embedding(n_items, emb_dim)
        self.mlp_user_emb = nn.Embedding(n_users, emb_dim)
        self.mlp_item_emb = nn.Embedding(n_items, emb_dim)
        layers, in_dim = [], emb_dim * 2
        for out_dim in mlp_dims:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.output_layer = nn.Linear(emb_dim + mlp_dims[-1], 1)  # 32+16=48

    def forward(self, user_ids, item_ids):
        mf_out  = self.mf_user_emb(user_ids) * self.mf_item_emb(item_ids)
        mlp_in  = torch.cat([self.mlp_user_emb(user_ids),
                              self.mlp_item_emb(item_ids)], dim=1)
        mlp_out = self.mlp(mlp_in)
        out     = self.output_layer(torch.cat([mf_out, mlp_out], dim=1))
        return torch.sigmoid(out).squeeze(1)

model_ncf = NCF_orig(n_users=6040, n_items=3706, emb_dim=32).to(device)
model_ncf.load_state_dict(torch.load("../data/ncf_best_v2.pth", map_location=device))
model_ncf.eval()
print("NCF 加载成功")
print(f"output_layer 输入维度验证: {model_ncf.output_layer.weight.shape}")  # 期望 [1, 48]

NCF 加载成功
output_layer 输入维度验证: torch.Size([1, 48])


In [21]:
# Cell 14：cold/warm 分组评估 NCF vs Two-Tower

def evaluate_by_coldness(model, model_type, val_df, train_df,
                          user2idx, movie2row, item2idx, user_attr2, device,
                          k=10, n_neg=99, random_state=42):
    val_last  = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
    train_pos = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
    train_cnt = train_df.groupby("user_id").size()

    if model_type == "ncf":
        valid_mids = list(item2idx.keys())   # 3706 部，Week 8 训练时见过的
    else:
        valid_mids = list(movie2row.keys())  # 3883 部

    results = {"cold": {"hrs":[], "ndcgs":[]},
               "warm": {"hrs":[], "ndcgs":[]}}

    eval_users = val_last[val_last["user_id"].isin(user2idx)].sample(
        min(500, len(val_last)), random_state=random_state)

    with torch.no_grad():
        for _, row in eval_users.iterrows():
            uid, pos_mid = row["user_id"], row["movie_id"]

            if model_type == "ncf":
                if pos_mid not in item2idx:
                    continue
                seen = train_pos.get(uid, set()) | {pos_mid}
                candidates = [m for m in valid_mids if m not in seen]
                neg_sample = np.random.choice(
                    candidates, size=min(n_neg, len(candidates)), replace=False).tolist()
                all_mids = [pos_mid] + neg_sample
                B = len(all_mids)
                u_t = torch.tensor([user2idx[uid]] * B, dtype=torch.long).to(device)
                i_t = torch.tensor([item2idx[m] for m in all_mids],
                                   dtype=torch.long).to(device)
                scores = model(u_t, i_t).cpu().numpy()
            else:
                if pos_mid not in movie2row:
                    continue
                seen = train_pos.get(uid, set()) | {pos_mid}
                candidates = [m for m in valid_mids if m not in seen]
                neg_sample = np.random.choice(
                    candidates, size=min(n_neg, len(candidates)), replace=False).tolist()
                all_mids = [pos_mid] + neg_sample
                B = len(all_mids)
                attr = user_attr2[uid]
                u_t = torch.tensor([user2idx[uid]] * B, dtype=torch.long).to(device)
                i_t = torch.tensor([movie2row[m] for m in all_mids],
                                   dtype=torch.long).to(device)
                g_t = torch.full((B,), attr["gender_idx"], dtype=torch.float32).to(device)
                a_t = torch.full((B,), attr["age"],        dtype=torch.float32).to(device)
                o_t = torch.full((B,), attr["occupation"], dtype=torch.long).to(device)
                scores = model(u_t, i_t, g_t, a_t, o_t).cpu().numpy()

            rank = (scores > scores[0]).sum() + 1
            hr   = 1.0 if rank <= k else 0.0
            ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0.0

            group = "cold" if train_cnt.get(uid, 0) < 5 else "warm"
            results[group]["hrs"].append(hr)
            results[group]["ndcgs"].append(ndcg)

    return results

res_ncf = evaluate_by_coldness(model_ncf, "ncf", val_df, train_df,
                                user2idx, movie2row, item2idx, user_attr2, device)
res_tt  = evaluate_by_coldness(model_tt,  "tt",  val_df, train_df,
                                user2idx, movie2row, item2idx, user_attr2, device)

print(f"\n{'':12} {'── Cold (train<5) ──':^24} {'── Warm (train≥5) ──':^24}")
print(f"{'模型':<12} {'HR@10':>10} {'NDCG@10':>10}  {'HR@10':>10} {'NDCG@10':>10}  {'cold N':>6} {'warm N':>6}")
print("-" * 80)
for name, res in [("NCF(W8)", res_ncf), ("Two-Tower", res_tt)]:
    c, w = res["cold"], res["warm"]
    print(f"{name:<12} "
          f"{np.mean(c['hrs']):>10.4f} {np.mean(c['ndcgs']):>10.4f}  "
          f"{np.mean(w['hrs']):>10.4f} {np.mean(w['ndcgs']):>10.4f}  "
          f"{len(c['hrs']):>6} {len(w['hrs']):>6}")


               ── Cold (train<5) ──     ── Warm (train≥5) ──  
模型                HR@10    NDCG@10       HR@10    NDCG@10  cold N warm N
--------------------------------------------------------------------------------
NCF(W8)          0.0588     0.0229      0.0895     0.0428     187    313
Two-Tower        0.3904     0.2021      0.4185     0.2152     187    313


In [22]:
# 快速验证：item2idx 和 movie2row 的 key 是否一致
item2idx_keys = set(item2idx.keys())
movie2row_keys = set(movie2row.keys())

print(f"item2idx  电影数: {len(item2idx_keys)}")   # 3706，有评分的
print(f"movie2row 电影数: {len(movie2row_keys)}")  # 3883，全部

only_in_movie2row = movie2row_keys - item2idx_keys
print(f"只在 movie2row 里（无评分电影）: {len(only_in_movie2row)}")

# 关键：item2idx 的值域
print(f"\nitem2idx 值域: [{min(item2idx.values())}, {max(item2idx.values())}]")

# 用一个已知电影验证两套 index 的差异
test_mid = 1  # Toy Story
print(f"\nmovie_id=1: item2idx={item2idx.get(1)}, movie2row={movie2row.get(1)}")
test_mid2 = 10
print(f"movie_id=10: item2idx={item2idx.get(10)}, movie2row={movie2row.get(10)}")

item2idx  电影数: 3706
movie2row 电影数: 3883
只在 movie2row 里（无评分电影）: 177

item2idx 值域: [0, 3705]

movie_id=1: item2idx=0, movie2row=0
movie_id=10: item2idx=9, movie2row=9


In [23]:
# 找一个 item2idx 和 movie2row 不一致的 movie_id
for mid in sorted(item2idx.keys()):
    if item2idx[mid] != movie2row.get(mid, -1):
        print(f"第一个不一致: movie_id={mid}, item2idx={item2idx[mid]}, movie2row={movie2row[mid]}")
        break

# 看看 movies 表里 movie_id 的空缺在哪
all_movie_ids_in_movies = sorted(movie2row.keys())
all_movie_ids_in_ratings = sorted(item2idx.keys())

# 找出只在 movies 里、不在 ratings 里的电影（无评分电影）
no_rating_mids = sorted(movie2row_keys - item2idx_keys)
print(f"\n前10个无评分电影 movie_id: {no_rating_mids[:10]}")
print(f"第一个无评分电影出现在位置: movie2row={movie2row[no_rating_mids[0]]}")

第一个不一致: movie_id=52, item2idx=50, movie2row=51

前10个无评分电影 movie_id: [np.int64(51), np.int64(109), np.int64(115), np.int64(143), np.int64(284), np.int64(285), np.int64(395), np.int64(399), np.int64(400), np.int64(403)]
第一个无评分电影出现在位置: movie2row=50


In [24]:
# Cell 15：修正 NCF 评估，候选集和 index 统一用 item2idx

def evaluate_ncf_correct(model, val_df, train_df, user2idx, item2idx,
                          device, k=10, n_neg=99, random_state=42):
    val_last  = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
    train_pos = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
    train_cnt = train_df.groupby("user_id").size()
    valid_mids = list(item2idx.keys())  # 3706，用 item2idx

    results = {"cold": {"hrs":[], "ndcgs":[]},
               "warm": {"hrs":[], "ndcgs":[]}}

    eval_users = val_last[val_last["user_id"].isin(user2idx)].sample(
        min(500, len(val_last)), random_state=random_state)

    with torch.no_grad():
        for _, row in eval_users.iterrows():
            uid, pos_mid = row["user_id"], row["movie_id"]
            if pos_mid not in item2idx:
                continue

            seen = train_pos.get(uid, set()) | {pos_mid}
            candidates = [m for m in valid_mids if m not in seen]
            neg_sample = np.random.choice(
                candidates, size=min(n_neg, len(candidates)), replace=False).tolist()
            all_mids = [pos_mid] + neg_sample
            B = len(all_mids)

            u_t = torch.tensor([user2idx[uid]] * B, dtype=torch.long).to(device)
            i_t = torch.tensor([item2idx[m] for m in all_mids],  # item2idx
                               dtype=torch.long).to(device)
            scores = model(u_t, i_t).cpu().numpy()

            rank = (scores > scores[0]).sum() + 1
            hr   = 1.0 if rank <= k else 0.0
            ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0.0

            group = "cold" if train_cnt.get(uid, 0) < 5 else "warm"
            results[group]["hrs"].append(hr)
            results[group]["ndcgs"].append(ndcg)

    return results

res_ncf = evaluate_ncf_correct(model_ncf, val_df, train_df,
                                user2idx, item2idx, device)
res_tt  = evaluate_by_coldness(model_tt, "tt", val_df, train_df,
                                user2idx, movie2row, item2idx, user_attr2, device)

print(f"\n{'':12} {'── Cold (train<5) ──':^24} {'── Warm (train≥5) ──':^24}")
print(f"{'模型':<12} {'HR@10':>10} {'NDCG@10':>10}  {'HR@10':>10} {'NDCG@10':>10}  {'cold N':>6} {'warm N':>6}")
print("-" * 80)
for name, res in [("NCF(W8)", res_ncf), ("Two-Tower", res_tt)]:
    c, w = res["cold"], res["warm"]
    print(f"{name:<12} "
          f"{np.mean(c['hrs']):>10.4f} {np.mean(c['ndcgs']):>10.4f}  "
          f"{np.mean(w['hrs']):>10.4f} {np.mean(w['ndcgs']):>10.4f}  "
          f"{len(c['hrs']):>6} {len(w['hrs']):>6}")


               ── Cold (train<5) ──     ── Warm (train≥5) ──  
模型                HR@10    NDCG@10       HR@10    NDCG@10  cold N warm N
--------------------------------------------------------------------------------
NCF(W8)          0.0588     0.0241      0.0863     0.0386     187    313
Two-Tower        0.3850     0.2061      0.4026     0.2110     187    313


In [25]:
# 最小化验证：取一个真实正样本，看模型给它的打分是否合理

# 找一个 warm 用户的正样本
sample_row = val_df[val_df["user_id"].isin(
    train_df.groupby("user_id").size()[lambda x: x >= 20].index
)].iloc[0]

uid     = sample_row["user_id"]
pos_mid = sample_row["movie_id"]
print(f"user_id={uid}, pos movie_id={pos_mid}")
print(f"user2idx[uid]={user2idx[uid]}, item2idx[pos_mid]={item2idx.get(pos_mid, 'NOT FOUND')}")

# 随机取 9 个负样本
valid_mids = list(item2idx.keys())
neg_mids = np.random.choice(valid_mids, size=9, replace=False).tolist()
all_mids = [pos_mid] + neg_mids

u_t = torch.tensor([user2idx[uid]] * 10, dtype=torch.long).to(device)
i_t = torch.tensor([item2idx[m] for m in all_mids], dtype=torch.long).to(device)

with torch.no_grad():
    scores = model_ncf(u_t, i_t).cpu().numpy()

print(f"\n正样本得分: {scores[0]:.4f}")
print(f"负样本得分: {scores[1:]}")
print(f"正样本排名: {(scores > scores[0]).sum() + 1} / 10")


user_id=1875, pos movie_id=1721
user2idx[uid]=1874, item2idx[pos_mid]=1574

正样本得分: 0.2053
负样本得分: [5.8223516e-01 3.9611449e-03 6.8073368e-01 2.4719298e-02 1.6049415e-01
 9.8040942e-03 1.8699691e-02 8.3689086e-02 3.1817908e-04]
正样本排名: 3 / 10


In [26]:
# 诊断：评估时有多少正样本被 continue 跳过
val_last = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
eval_users = val_last[val_last["user_id"].isin(user2idx)].sample(
    min(500, len(val_last)), random_state=42)

skipped = 0
kept = 0
for _, row in eval_users.iterrows():
    uid, pos_mid = row["user_id"], row["movie_id"]
    if pos_mid not in item2idx:
        skipped += 1
    else:
        kept += 1

print(f"评估样本总数: {len(eval_users)}")
print(f"正样本在 item2idx 里（kept）: {kept}")
print(f"正样本不在 item2idx 里（skipped）: {skipped}")

# 另外验证：Week 8 原始评估是怎么做的？
# Week 8 用的是什么 val 集？
print(f"\nval_df 总行数: {len(val_df)}")
print(f"val_last 行数: {len(val_last)}")
print(f"val_last 中 user 在 user2idx 里的数量: {val_last['user_id'].isin(user2idx).sum()}")


评估样本总数: 500
正样本在 item2idx 里（kept）: 500
正样本不在 item2idx 里（skipped）: 0

val_df 总行数: 200042
val_last 行数: 1783
val_last 中 user 在 user2idx 里的数量: 1783


In [27]:
# 还原 Week 8 评估协议，逐字检查
# 请打开 week8.ipynb，把评估函数（evaluate 或 evaluate_model 之类的）
# 完整贴给我

# 在此之前，先做一个最简单的 sanity check：
# 对同一个用户，Week 8 评估 vs 现在评估，候选集是否一样？

uid = 1875  # 刚才用过的用户
pos_mid = 1721

train_seen = train_df[train_df["user_id"] == uid]["movie_id"].tolist()
print(f"user {uid} 在 train 里看过 {len(train_seen)} 部电影")
print(f"pos_mid={pos_mid} 在 item2idx 里: {pos_mid in item2idx}")

# 用 item2idx 生成候选集
valid_mids = list(item2idx.keys())
seen = set(train_seen) | {pos_mid}
candidates = [m for m in valid_mids if m not in seen]
np.random.seed(42)
neg_sample = np.random.choice(candidates, size=99, replace=False).tolist()
all_mids = [pos_mid] + neg_sample

u_t = torch.tensor([user2idx[uid]] * 100, dtype=torch.long).to(device)
i_t = torch.tensor([item2idx[m] for m in all_mids], dtype=torch.long).to(device)

with torch.no_grad():
    scores = model_ncf(u_t, i_t).cpu().numpy()

rank = (scores > scores[0]).sum() + 1
print(f"正样本得分: {scores[0]:.4f}, 排名: {rank}/100")
print(f"Top5 得分: {sorted(scores, reverse=True)[:5]}")

user 1875 在 train 里看过 242 部电影
pos_mid=1721 在 item2idx 里: True
正样本得分: 0.2053, 排名: 29/100
Top5 得分: [np.float32(0.8819429), np.float32(0.78691304), np.float32(0.73725307), np.float32(0.68145037), np.float32(0.5914314)]


In [30]:
# Cell 16：干净重训 NCF，单一 index 体系
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

# ── 数据 ──────────────────────────────────────────────
ratings = pd.read_csv("../data/ml-1m/ratings.dat", sep="::",
    names=["user_id","movie_id","rating","timestamp"], engine="python")

ratings_sorted = ratings.sort_values("timestamp")
n = len(ratings_sorted)
train_df = ratings_sorted.iloc[:int(n * 0.8)].copy()
val_df   = ratings_sorted.iloc[int(n * 0.8):].copy()

# 单一 index：只用 movie_id 在 ratings 里出现过的电影
all_users  = sorted(ratings["user_id"].unique())
all_movies = sorted(ratings["movie_id"].unique())
user2idx = {u: i for i, u in enumerate(all_users)}
movie2idx = {m: i for i, m in enumerate(all_movies)}  # 唯一一套，全程用这个

n_users = len(user2idx)   # 6040
n_movies = len(movie2idx) # 3706
print(f"n_users={n_users}, n_movies={n_movies}")

# ── Dataset ───────────────────────────────────────────
user_pos_movies = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
all_movie_list  = list(movie2idx.keys())

class NCFDataset(Dataset):
    def __init__(self, df, user2idx, movie2idx, user_pos_movies, all_movie_list, neg_ratio=4):
        self.user2idx        = user2idx
        self.movie2idx       = movie2idx
        self.user_pos_movies = user_pos_movies
        self.all_movie_list  = all_movie_list
        self.neg_ratio       = neg_ratio
        self.pos_pairs       = list(zip(df["user_id"].values, df["movie_id"].values))

    def __len__(self):
        return len(self.pos_pairs) * (1 + self.neg_ratio)

    def __getitem__(self, idx):
        pos_idx = idx // (1 + self.neg_ratio)
        is_neg  = (idx %  (1 + self.neg_ratio)) != 0
        user_id, movie_id = self.pos_pairs[pos_idx]
        if is_neg:
            while True:
                neg = self.all_movie_list[np.random.randint(len(self.all_movie_list))]
                if neg not in self.user_pos_movies.get(user_id, set()):
                    movie_id = neg
                    break
            label = 0.0
        else:
            label = 1.0
        return (torch.tensor(self.user2idx[user_id],  dtype=torch.long),
                torch.tensor(self.movie2idx[movie_id], dtype=torch.long),
                torch.tensor(label, dtype=torch.float32))

train_ds = NCFDataset(train_df, user2idx, movie2idx, user_pos_movies, all_movie_list)
val_ds   = NCFDataset(val_df,   user2idx, movie2idx, user_pos_movies, all_movie_list)
train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=2048, shuffle=False, num_workers=0)
print(f"train={len(train_ds):,}  val={len(val_ds):,}")

# ── 模型 ──────────────────────────────────────────────
class NCF(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=32, mlp_dims=[64,32,16]):
        super().__init__()
        self.mf_user_emb  = nn.Embedding(n_users,  emb_dim)
        self.mf_movie_emb = nn.Embedding(n_movies, emb_dim)
        self.mlp_user_emb  = nn.Embedding(n_users,  emb_dim)
        self.mlp_movie_emb = nn.Embedding(n_movies, emb_dim)
        layers, in_dim = [], emb_dim * 2
        for out_dim in mlp_dims:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(emb_dim + mlp_dims[-1], 1)
        for emb in [self.mf_user_emb, self.mf_movie_emb,
                    self.mlp_user_emb, self.mlp_movie_emb]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, u, m):
        mf  = self.mf_user_emb(u) * self.mf_movie_emb(m)
        mlp = self.mlp(torch.cat([self.mlp_user_emb(u), self.mlp_movie_emb(m)], dim=1))
        return torch.sigmoid(self.out(torch.cat([mf, mlp], dim=1))).squeeze(1)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model  = NCF(n_users, n_movies).to(device)
opt    = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
crit   = nn.BCELoss()

# ── 评估 HR@10 ────────────────────────────────────────
def evaluate_hr(model, val_df, train_df, user2idx, movie2idx,
                device, k=10, n_neg=99, n_eval=500, seed=42):
    model.eval()
    val_last  = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
    train_pos = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
    train_cnt = train_df.groupby("user_id").size()
    all_mids  = list(movie2idx.keys())

    hrs, ndcgs = [], []
    rows = val_last[val_last["user_id"].isin(user2idx) &
                    val_last["movie_id"].isin(movie2idx)].sample(
                    min(n_eval, len(val_last)), random_state=seed)

    with torch.no_grad():
        for _, row in rows.iterrows():
            uid, pos_mid = row["user_id"], row["movie_id"]
            seen = train_pos.get(uid, set()) | {pos_mid}
            cands = [m for m in all_mids if m not in seen]
            negs  = np.random.choice(cands, size=min(n_neg, len(cands)),
                                     replace=False).tolist()
            mids  = [pos_mid] + negs
            B     = len(mids)
            u_t   = torch.tensor([user2idx[uid]] * B, dtype=torch.long).to(device)
            m_t   = torch.tensor([movie2idx[m]  for m in mids], dtype=torch.long).to(device)
            scores = model(u_t, m_t).cpu().numpy()
            rank   = (scores > scores[0]).sum() + 1
            hrs.append(1.0 if rank <= k else 0.0)
            ndcgs.append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)

    return np.mean(hrs), np.mean(ndcgs)

# ── 训练循环 ──────────────────────────────────────────
def run_epoch(model, loader, opt, crit, device, train=True):
    model.train() if train else model.eval()
    total_loss, total_n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for u, m, lbl in loader:
            u, m, lbl = u.to(device), m.to(device), lbl.to(device)
            if train:
                opt.zero_grad()
            pred = model(u, m)
            loss = crit(pred, lbl)
            if train:
                loss.backward()
                opt.step()
            total_loss += loss.item() * len(lbl)
            total_n    += len(lbl)
    return total_loss / total_n

best_hr, best_epoch = 0.0, -1
print(f"\n{'Epoch':>5} {'Train':>10} {'Val':>10} {'HR@10':>8} {'NDCG@10':>9}")
print("-" * 48)

for epoch in range(1, 16):
    tr  = run_epoch(model, train_loader, opt, crit, device, train=True)
    val = run_epoch(model, val_loader,   opt, crit, device, train=False)
    hr, ndcg = evaluate_hr(model, val_df, train_df, user2idx, movie2idx, device)
    marker = ""
    if hr > best_hr:
        best_hr, best_epoch = hr, epoch
        torch.save({
            "model_state": model.state_dict(),
            "hparams": {"emb_dim": 32, "mlp_dims": [64,32,16],
                        "n_users": n_users, "n_movies": n_movies}
        }, "../data/ncf_clean.pth")
        marker = " ✓"
    print(f"{epoch:>5} {tr:>10.6f} {val:>10.6f} {hr:>8.4f} {ndcg:>9.4f}{marker}")

print(f"\n最佳 HR@10: {best_hr:.4f}（epoch {best_epoch}）")

n_users=6040, n_movies=3706
train=4,000,835  val=1,000,210

Epoch      Train        Val    HR@10   NDCG@10
------------------------------------------------
    1   0.374462   0.400461   0.3760    0.2047 ✓
    2   0.352957   0.406367   0.3640    0.1945
    3   0.327356   0.410069   0.4060    0.2011 ✓
    4   0.312376   0.415645   0.3840    0.1848
    5   0.301137   0.421393   0.3800    0.1948
    6   0.292650   0.421421   0.3920    0.2029
    7   0.286933   0.426199   0.3880    0.2036
    8   0.282754   0.432052   0.4120    0.2140 ✓
    9   0.279643   0.428600   0.4060    0.2066
   10   0.276913   0.427430   0.3800    0.1954
   11   0.274203   0.430642   0.4000    0.2067
   12   0.272339   0.430030   0.4080    0.2110
   13   0.270545   0.435148   0.4080    0.2057
   14   0.269354   0.436186   0.4120    0.2122
   15   0.267852   0.435167   0.3880    0.1987

最佳 HR@10: 0.4120（epoch 8）


In [29]:
# Cell 17：Two-Tower v2，emb_dim=32，tower 逐层压缩

class TwoTowerV2(nn.Module):
    def __init__(self, n_users, n_movies, genre_matrix,
                 emb_dim=32, tower_dims=[64, 32]):
        super().__init__()
        self.register_buffer("genre_matrix", genre_matrix)  # [3883, 18]
        n_genres = genre_matrix.shape[1]  # 18

        # age 归一化
        age_raw = torch.tensor([1,18,25,35,45,50,56], dtype=torch.float32)
        self.register_buffer("age_min",   age_raw.min())
        self.register_buffer("age_range", age_raw.max() - age_raw.min())

        # User Tower：emb(32) + occupation_emb(16) + gender(1) + age(1) = 50
        self.user_emb  = nn.Embedding(n_users,  emb_dim)
        self.occ_emb   = nn.Embedding(21, 16)
        user_in = emb_dim + 16 + 1 + 1  # 50

        # Item Tower：emb(32) + genre(18) = 50
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        item_in = emb_dim + n_genres     # 50

        # 两个 tower 结构对称，逐层压缩
        def make_tower(in_dim, dims):
            layers, d = [], in_dim
            for out_d in dims:
                layers += [nn.Linear(d, out_d), nn.ReLU(), nn.Dropout(0.2)]
                d = out_d
            return nn.Sequential(*layers)

        self.user_tower  = make_tower(user_in, tower_dims)
        self.movie_tower = make_tower(item_in, tower_dims)

        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, user_ids, movie_ids, gender, age_raw, occ_ids):
        # User Tower
        age_n = (age_raw - self.age_min) / self.age_range  # [B]
        u_vec = self.user_tower(torch.cat([
            self.user_emb(user_ids),
            self.occ_emb(occ_ids),
            gender.unsqueeze(1),
            age_n.unsqueeze(1)
        ], dim=1))                                          # [B, 32]

        # Item Tower
        g_vec = self.genre_matrix[movie_ids]               # [B, 18]
        m_vec = self.movie_tower(torch.cat([
            self.movie_emb(movie_ids), g_vec
        ], dim=1))                                         # [B, 32]

        return torch.sigmoid((u_vec * m_vec).sum(dim=1))  # [B]

# 验证结构
model_tt2 = TwoTowerV2(n_users=n_users, n_movies=n_movies,
                        genre_matrix=genre_tensor).to(device)
print(f"参数量: {sum(p.numel() for p in model_tt2.parameters()):,}")
print(f"NCF 参数量:        {sum(p.numel() for p in model.parameters()):,}")

_u = torch.tensor([0,1,2], dtype=torch.long).to(device)
_m = torch.tensor([0,1,2], dtype=torch.long).to(device)
_g = torch.tensor([0.,1.,0.]).to(device)
_a = torch.tensor([25.,35.,18.]).to(device)
_o = torch.tensor([7,0,4], dtype=torch.long).to(device)
print(f"forward 输出: {model_tt2(_u, _m, _g, _a, _o)}")

参数量: 322,896
NCF 参数量:        630,561
forward 输出: tensor([0.5088, 0.5046, 0.5134], device='mps:0', grad_fn=<SigmoidBackward0>)


In [31]:
# Cell 18：训练 TwoTowerV2

# 重建 user_attr（用原始 age 值）
users_df = pd.read_csv("../data/ml-1m/users.dat", sep="::",
    names=["user_id","gender","age","occupation","zip"], engine="python")
user_attr = {}
for _, row in users_df.iterrows():
    user_attr[row["user_id"]] = {
        "gender": float(row["gender"] == "M"),
        "age":    float(row["age"]),
        "occ":    int(row["occupation"])
    }

# genre_matrix：movie_idx（movie2idx 的值）→ genre 向量
# 注意：Two-Tower 现在统一用 movie2idx（和 NCF 同一套）
# genre_matrix 行号必须和 movie2idx 对齐
genre_matrix = torch.zeros(n_movies, 18, dtype=torch.float32)
genre_list = ['Action','Adventure','Animation',"Children's",'Comedy',
              'Crime','Documentary','Drama','Fantasy','Film-Noir',
              'Horror','Musical','Mystery','Romance','Sci-Fi',
              'Thriller','War','Western']
g2i = {g: i for i, g in enumerate(genre_list)}

movies_df = pd.read_csv("../data/ml-1m/movies.dat", sep="::",
    names=["movie_id","title","genres"], engine="python", encoding="latin-1")
for _, row in movies_df.iterrows():
    mid = row["movie_id"]
    if mid not in movie2idx:
        continue  # 跳过无评分电影
    idx = movie2idx[mid]
    for tag in row["genres"].split("|"):
        if tag in g2i:
            genre_matrix[idx, g2i[tag]] = 1.0

print(f"genre_matrix shape: {genre_matrix.shape}")  # [3706, 18]
zero_rows = (genre_matrix.sum(dim=1) == 0).sum().item()
print(f"全零行（无 genre）: {zero_rows}")

# Dataset
class TTDataset(Dataset):
    def __init__(self, df, user2idx, movie2idx, user_attr,
                 user_pos_movies, all_movie_list, neg_ratio=4):
        self.user2idx        = user2idx
        self.movie2idx       = movie2idx
        self.user_attr       = user_attr
        self.user_pos_movies = user_pos_movies
        self.all_movie_list  = all_movie_list
        self.neg_ratio       = neg_ratio
        self.pos_pairs       = list(zip(df["user_id"].values, df["movie_id"].values))

    def __len__(self):
        return len(self.pos_pairs) * (1 + self.neg_ratio)

    def __getitem__(self, idx):
        pos_idx = idx // (1 + self.neg_ratio)
        is_neg  = (idx %  (1 + self.neg_ratio)) != 0
        user_id, movie_id = self.pos_pairs[pos_idx]
        if is_neg:
            while True:
                neg = self.all_movie_list[np.random.randint(len(self.all_movie_list))]
                if neg not in self.user_pos_movies.get(user_id, set()):
                    movie_id = neg
                    break
            label = 0.0
        else:
            label = 1.0
        attr = self.user_attr[user_id]
        return (
            torch.tensor(self.user2idx[user_id],   dtype=torch.long),
            torch.tensor(self.movie2idx[movie_id], dtype=torch.long),
            torch.tensor(attr["gender"],           dtype=torch.float32),
            torch.tensor(attr["age"],              dtype=torch.float32),
            torch.tensor(attr["occ"],              dtype=torch.long),
            torch.tensor(label,                    dtype=torch.float32)
        )

train_ds_tt = TTDataset(train_df, user2idx, movie2idx, user_attr,
                         user_pos_movies, all_movie_list)
val_ds_tt   = TTDataset(val_df,   user2idx, movie2idx, user_attr,
                         user_pos_movies, all_movie_list)
train_loader_tt = DataLoader(train_ds_tt, batch_size=2048, shuffle=True,  num_workers=0)
val_loader_tt   = DataLoader(val_ds_tt,   batch_size=2048, shuffle=False, num_workers=0)
print(f"train={len(train_ds_tt):,}  val={len(val_ds_tt):,}")

# 评估函数（和 NCF 版本对称，同一套 movie2idx）
def evaluate_hr_tt(model, val_df, train_df, user2idx, movie2idx,
                   user_attr, device, k=10, n_neg=99, n_eval=500, seed=42):
    model.eval()
    val_last  = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
    train_pos = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
    all_mids  = list(movie2idx.keys())
    hrs, ndcgs = [], []
    rows = val_last[val_last["user_id"].isin(user2idx) &
                    val_last["movie_id"].isin(movie2idx)].sample(
                    min(n_eval, len(val_last)), random_state=seed)
    with torch.no_grad():
        for _, row in rows.iterrows():
            uid, pos_mid = row["user_id"], row["movie_id"]
            seen  = train_pos.get(uid, set()) | {pos_mid}
            cands = [m for m in all_mids if m not in seen]
            negs  = np.random.choice(cands, size=min(n_neg, len(cands)),
                                     replace=False).tolist()
            mids  = [pos_mid] + negs
            B     = len(mids)
            attr  = user_attr[uid]
            u_t   = torch.tensor([user2idx[uid]] * B, dtype=torch.long).to(device)
            m_t   = torch.tensor([movie2idx[m] for m in mids], dtype=torch.long).to(device)
            g_t   = torch.full((B,), attr["gender"], dtype=torch.float32).to(device)
            a_t   = torch.full((B,), attr["age"],    dtype=torch.float32).to(device)
            o_t   = torch.full((B,), attr["occ"],    dtype=torch.long).to(device)
            scores = model(u_t, m_t, g_t, a_t, o_t).cpu().numpy()
            rank   = (scores > scores[0]).sum() + 1
            hrs.append(1.0 if rank <= k else 0.0)
            ndcgs.append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)
    return np.mean(hrs), np.mean(ndcgs)

# 训练
model_tt2 = TwoTowerV2(n_users=n_users, n_movies=n_movies,
                        genre_matrix=genre_matrix).to(device)
opt_tt2 = optim.Adam(model_tt2.parameters(), lr=1e-3, weight_decay=1e-5)

best_hr_tt2, best_epoch_tt2 = 0.0, -1
print(f"\n{'Epoch':>5} {'Train':>10} {'Val':>10} {'HR@10':>8} {'NDCG@10':>9}")
print("-" * 48)

for epoch in range(1, 16):
    # train
    model_tt2.train()
    total_loss, total_n = 0.0, 0
    for u, m, g, a, o, lbl in train_loader_tt:
        u,m,g,a,o,lbl = (x.to(device) for x in (u,m,g,a,o,lbl))
        opt_tt2.zero_grad()
        pred = model_tt2(u, m, g, a, o)
        loss = crit(pred, lbl)
        loss.backward()
        opt_tt2.step()
        total_loss += loss.item() * len(lbl)
        total_n    += len(lbl)
    tr = total_loss / total_n

    # val loss
    model_tt2.eval()
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for u, m, g, a, o, lbl in val_loader_tt:
            u,m,g,a,o,lbl = (x.to(device) for x in (u,m,g,a,o,lbl))
            pred = model_tt2(u, m, g, a, o)
            loss = crit(pred, lbl)
            total_loss += loss.item() * len(lbl)
            total_n    += len(lbl)
    val = total_loss / total_n

    hr, ndcg = evaluate_hr_tt(model_tt2, val_df, train_df,
                               user2idx, movie2idx, user_attr, device)
    marker = ""
    if hr > best_hr_tt2:
        best_hr_tt2, best_epoch_tt2 = hr, epoch
        torch.save({
            "model_state": model_tt2.state_dict(),
            "hparams": {"emb_dim": 32, "tower_dims": [64,32],
                        "n_users": n_users, "n_movies": n_movies}
        }, "../data/two_tower_v2_best.pth")
        marker = " ✓"
    print(f"{epoch:>5} {tr:>10.6f} {val:>10.6f} {hr:>8.4f} {ndcg:>9.4f}{marker}")

print(f"\n最佳 HR@10: {best_hr_tt2:.4f}（epoch {best_epoch_tt2}）")

genre_matrix shape: torch.Size([3706, 18])
全零行（无 genre）: 0
train=4,000,835  val=1,000,210

Epoch      Train        Val    HR@10   NDCG@10
------------------------------------------------
    1   0.693200   0.693147   1.0000    1.0000 ✓
    2   0.693147   0.693147   1.0000    1.0000
    3   0.693147   0.693147   1.0000    1.0000
    4   0.693147   0.693147   1.0000    1.0000
    5   0.693147   0.693147   1.0000    1.0000
    6   0.693147   0.693147   1.0000    1.0000
    7   0.693147   0.693147   1.0000    1.0000
    8   0.693147   0.693147   1.0000    1.0000
    9   0.693147   0.693147   1.0000    1.0000
   10   0.693147   0.693147   1.0000    1.0000
   11   0.693147   0.693147   1.0000    1.0000
   12   0.693147   0.693147   1.0000    1.0000
   13   0.693147   0.693147   1.0000    1.0000
   14   0.693147   0.693147   1.0000    1.0000
   15   0.693147   0.693147   1.0000    1.0000

最佳 HR@10: 1.0000（epoch 1）


In [32]:
# 快速诊断：检查模型内部的 genre_matrix 是否正常
print(f"genre_matrix（外部）sum: {genre_matrix.sum():.1f}")
print(f"model_tt2.genre_matrix sum: {model_tt2.genre_matrix.sum():.1f}")
print(f"model_tt2.genre_matrix device: {model_tt2.genre_matrix.device}")

# 取第一个 batch 手动跑 forward，看中间值
u, m, g, a, o, lbl = next(iter(train_loader_tt))
u,m,g,a,o,lbl = (x.to(device) for x in (u,m,g,a,o,lbl))

with torch.no_grad():
    # 检查 genre_matrix 取出来的值
    gvec = model_tt2.genre_matrix[m[:5]]
    print(f"\ngenre_matrix[m[:5]] sum per row: {gvec.sum(dim=1)}")
    
    # 检查 forward 输出
    out = model_tt2(u[:5], m[:5], g[:5], a[:5], o[:5])
    print(f"forward 输出: {out}")

genre_matrix（外部）sum: 6192.0
model_tt2.genre_matrix sum: 6192.0
model_tt2.genre_matrix device: mps:0

genre_matrix[m[:5]] sum per row: tensor([2., 2., 2., 1., 1.], device='mps:0')
forward 输出: tensor([0.5000, 0.5000, 0.5000, 0.5000, 0.5000], device='mps:0')


In [33]:
# 检查两个 tower 的输出分布
with torch.no_grad():
    age_n = (a[:8] - model_tt2.age_min) / model_tt2.age_range
    u_in = torch.cat([
        model_tt2.user_emb(u[:8]),
        model_tt2.occ_emb(o[:8]),
        g[:8].unsqueeze(1),
        age_n.unsqueeze(1)
    ], dim=1)
    m_in = torch.cat([
        model_tt2.movie_emb(m[:8]),
        model_tt2.genre_matrix[m[:8]]
    ], dim=1)
    
    u_vec = model_tt2.user_tower(u_in)
    m_vec = model_tt2.movie_tower(m_in)
    dot   = (u_vec * m_vec).sum(dim=1)
    
    print(f"u_vec 均值: {u_vec.mean():.6f}, std: {u_vec.std():.6f}")
    print(f"m_vec 均值: {m_vec.mean():.6f}, std: {m_vec.std():.6f}")
    print(f"dot product: {dot}")

u_vec 均值: 0.000000, std: 0.000000
m_vec 均值: 0.000000, std: 0.000000
dot product: tensor([0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')


In [34]:
# Cell 19：修复 TwoTowerV2 初始化

class TwoTowerV2(nn.Module):
    def __init__(self, n_users, n_movies, genre_matrix,
                 emb_dim=32, tower_dims=[64, 32]):
        super().__init__()
        self.register_buffer("genre_matrix", genre_matrix)
        n_genres = genre_matrix.shape[1]

        age_raw = torch.tensor([1,18,25,35,45,50,56], dtype=torch.float32)
        self.register_buffer("age_min",   age_raw.min())
        self.register_buffer("age_range", age_raw.max() - age_raw.min())

        self.user_emb  = nn.Embedding(n_users,  emb_dim)
        self.occ_emb   = nn.Embedding(21, 16)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)

        user_in = emb_dim + 16 + 1 + 1  # 50
        item_in = emb_dim + n_genres     # 50

        def make_tower(in_dim, dims):
            layers, d = [], in_dim
            for i, out_d in enumerate(dims):
                layers.append(nn.Linear(d, out_d))
                if i < len(dims) - 1:      # 最后一层不加 ReLU/Dropout
                    layers.append(nn.ReLU())
                    layers.append(nn.Dropout(0.2))
                d = out_d
            return nn.Sequential(*layers)

        self.user_tower  = make_tower(user_in, tower_dims)
        self.movie_tower = make_tower(item_in, tower_dims)

        # kaiming 初始化：专为 ReLU 设计，避免 dying ReLU
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, user_ids, movie_ids, gender, age_raw, occ_ids):
        age_n = (age_raw - self.age_min) / self.age_range
        u_vec = self.user_tower(torch.cat([
            self.user_emb(user_ids),
            self.occ_emb(occ_ids),
            gender.unsqueeze(1),
            age_n.unsqueeze(1)
        ], dim=1))

        m_vec = self.movie_tower(torch.cat([
            self.movie_emb(movie_ids),
            self.genre_matrix[movie_ids]
        ], dim=1))

        return torch.sigmoid((u_vec * m_vec).sum(dim=1))

# 验证输出不再是 0.5
model_tt2 = TwoTowerV2(n_users=n_users, n_movies=n_movies,
                        genre_matrix=genre_matrix).to(device)

u, m, g, a, o, lbl = next(iter(train_loader_tt))
u,m,g,a,o,lbl = (x.to(device) for x in (u,m,g,a,o,lbl))
with torch.no_grad():
    out = model_tt2(u[:8], m[:8], g[:8], a[:8], o[:8])
    print(f"修复后 forward 输出: {out}")
    print(f"是否全是 0.5: {(out == 0.5).all().item()}")

修复后 forward 输出: tensor([0.5752, 0.2400, 0.3197, 0.1986, 0.4861, 0.4459, 0.4435, 0.3255],
       device='mps:0')
是否全是 0.5: False


In [35]:
# Cell 20：训练修复后的 TwoTowerV2
import time

opt_tt2 = optim.Adam(model_tt2.parameters(), lr=1e-3, weight_decay=1e-5)

best_hr_tt2, best_epoch_tt2 = 0.0, -1
print(f"{'Epoch':>5} {'Train':>10} {'Val':>10} {'HR@10':>8} {'NDCG@10':>9} {'Time':>6}")
print("-" * 55)

for epoch in range(1, 16):
    t0 = time.time()
    
    model_tt2.train()
    total_loss, total_n = 0.0, 0
    for u, m, g, a, o, lbl in train_loader_tt:
        u,m,g,a,o,lbl = (x.to(device) for x in (u,m,g,a,o,lbl))
        opt_tt2.zero_grad()
        pred = model_tt2(u, m, g, a, o)
        loss = crit(pred, lbl)
        loss.backward()
        opt_tt2.step()
        total_loss += loss.item() * len(lbl)
        total_n    += len(lbl)
    tr = total_loss / total_n

    model_tt2.eval()
    total_loss, total_n = 0.0, 0
    with torch.no_grad():
        for u, m, g, a, o, lbl in val_loader_tt:
            u,m,g,a,o,lbl = (x.to(device) for x in (u,m,g,a,o,lbl))
            pred = model_tt2(u, m, g, a, o)
            loss = crit(pred, lbl)
            total_loss += loss.item() * len(lbl)
            total_n    += len(lbl)
    val = total_loss / total_n

    hr, ndcg = evaluate_hr_tt(model_tt2, val_df, train_df,
                               user2idx, movie2idx, user_attr, device)
    elapsed = time.time() - t0

    marker = ""
    if hr > best_hr_tt2:
        best_hr_tt2, best_epoch_tt2 = hr, epoch
        torch.save({
            "model_state": model_tt2.state_dict(),
            "hparams": {"emb_dim": 32, "tower_dims": [64,32],
                        "n_users": n_users, "n_movies": n_movies}
        }, "../data/two_tower_v2_best.pth")
        marker = " ✓"

    print(f"{epoch:>5} {tr:>10.6f} {val:>10.6f} {hr:>8.4f} {ndcg:>9.4f} {elapsed:>5.0f}s{marker}")

print(f"\n最佳 HR@10: {best_hr_tt2:.4f}（epoch {best_epoch_tt2}）")


Epoch      Train        Val    HR@10   NDCG@10   Time
-------------------------------------------------------
    1   0.337946   0.398419   0.4080    0.2171   116s ✓
    2   0.299040   0.400334   0.4160    0.2128   116s ✓
    3   0.288382   0.404369   0.4280    0.2222   114s ✓
    4   0.284074   0.406467   0.4180    0.2139   114s
    5   0.281096   0.407110   0.4080    0.2120   112s
    6   0.279378   0.406623   0.4180    0.2202  1013s
    7   0.277530   0.408806   0.4160    0.2132  2807s
    8   0.276500   0.407259   0.4260    0.2161   113s
    9   0.275472   0.409075   0.4180    0.2235   112s
   10   0.274495   0.407057   0.4180    0.2202  1013s
   11   0.273829   0.407365   0.4180    0.2233   107s
   12   0.273395   0.409732   0.4080    0.2191   107s
   13   0.272631   0.409320   0.4120    0.2196   184s
   14   0.272114   0.412323   0.4160    0.2330   115s
   15   0.271701   0.410774   0.4060    0.2219   114s

最佳 HR@10: 0.4280（epoch 3）


In [36]:
# Cell 21：构建固定评估集 + 最终对比

# 一次性构建固定评估集，所有模型共用
def build_fixed_eval_set(val_df, train_df, movie2idx, user2idx,
                          n_eval=500, n_neg=99, seed=42):
    np.random.seed(seed)
    val_last  = val_df.sort_values("timestamp").groupby("user_id").last().reset_index()
    train_pos = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
    all_mids  = list(movie2idx.keys())

    rows = val_last[val_last["user_id"].isin(user2idx) &
                    val_last["movie_id"].isin(movie2idx)].sample(
                    min(n_eval, len(val_last)), random_state=seed)

    eval_set = []  # list of (uid, pos_mid, [neg_mid x 99])
    for _, row in rows.iterrows():
        uid, pos_mid = row["user_id"], row["movie_id"]
        seen  = train_pos.get(uid, set()) | {pos_mid}
        cands = [m for m in all_mids if m not in seen]
        negs  = np.random.choice(cands, size=min(n_neg, len(cands)),
                                 replace=False).tolist()
        eval_set.append((uid, pos_mid, negs))

    print(f"固定评估集构建完成：{len(eval_set)} 个用户，每人 1正+{n_neg}负")
    return eval_set

fixed_eval = build_fixed_eval_set(val_df, train_df, movie2idx, user2idx)

# 用固定评估集评估任意模型
def eval_fixed(model, model_type, fixed_eval, user2idx, movie2idx,
               user_attr, device, k=10):
    model.eval()
    hrs, ndcgs = [], []
    with torch.no_grad():
        for uid, pos_mid, negs in fixed_eval:
            mids = [pos_mid] + negs
            B    = len(mids)
            u_t  = torch.tensor([user2idx[uid]] * B, dtype=torch.long).to(device)
            m_t  = torch.tensor([movie2idx[m] for m in mids], dtype=torch.long).to(device)

            if model_type == "ncf":
                scores = model(u_t, m_t).cpu().numpy()
            else:
                attr = user_attr[uid]
                g_t  = torch.full((B,), attr["gender"], dtype=torch.float32).to(device)
                a_t  = torch.full((B,), attr["age"],    dtype=torch.float32).to(device)
                o_t  = torch.full((B,), attr["occ"],    dtype=torch.long).to(device)
                scores = model(u_t, m_t, g_t, a_t, o_t).cpu().numpy()

            rank = (scores > scores[0]).sum() + 1
            hrs.append(1.0 if rank <= k else 0.0)
            ndcgs.append(1.0 / np.log2(rank + 1) if rank <= k else 0.0)

    return np.mean(hrs), np.mean(ndcgs)

# 加载各自最佳权重
model_ncf.load_state_dict(
    torch.load("../data/ncf_best_v2.pth", map_location=device))

ckpt = torch.load("../data/two_tower_v2_best.pth", map_location=device)
model_tt2.load_state_dict(ckpt["model_state"])

# 统一评估
hr_ncf,  ndcg_ncf  = eval_fixed(model_ncf,  "ncf", fixed_eval,
                                  user2idx, movie2idx, user_attr, device)
hr_tt2,  ndcg_tt2  = eval_fixed(model_tt2,  "tt",  fixed_eval,
                                  user2idx, movie2idx, user_attr, device)

print(f"\n{'模型':<20} {'HR@10':>8} {'NDCG@10':>10}  备注")
print("-" * 55)
print(f"{'随机基准':<20} {'0.1000':>8} {'—':>10}")
print(f"{'NCF v2 (Week8)':<20} {hr_ncf:>8.4f} {ndcg_ncf:>10.4f}  emb=32, mlp=[64,32,16]")
print(f"{'Two-Tower v2':<20} {hr_tt2:>8.4f} {ndcg_tt2:>10.4f}  emb=32, tower=[64,32]")

固定评估集构建完成：500 个用户，每人 1正+99负

模型                      HR@10    NDCG@10  备注
-------------------------------------------------------
随机基准                   0.1000          —
NCF v2 (Week8)         0.0640     0.0303  emb=32, mlp=[64,32,16]
Two-Tower v2           0.4100     0.2185  emb=32, tower=[64,32]


In [37]:
# 加载 ncf_clean.pth（Cell 16 重训的，和 movie2idx 对齐的版本）
ckpt_clean = torch.load("../data/ncf_clean.pth", map_location=device)
print("hparams:", ckpt_clean["hparams"])

model_ncf_clean = NCF(
    n_users  = ckpt_clean["hparams"]["n_users"],
    n_movies = ckpt_clean["hparams"]["n_movies"],
    emb_dim  = ckpt_clean["hparams"]["emb_dim"],
    mlp_dims = ckpt_clean["hparams"]["mlp_dims"]
).to(device)
model_ncf_clean.load_state_dict(ckpt_clean["model_state"])
model_ncf_clean.eval()
print("ncf_clean 加载成功")

hr_ncf_clean, ndcg_ncf_clean = eval_fixed(
    model_ncf_clean, "ncf", fixed_eval, user2idx, movie2idx, user_attr, device)

print(f"\n{'模型':<22} {'HR@10':>8} {'NDCG@10':>10}  备注")
print("-" * 58)
print(f"{'随机基准':<22} {'0.1000':>8} {'—':>10}")
print(f"{'NCF clean':<22} {hr_ncf_clean:>8.4f} {ndcg_ncf_clean:>10.4f}  Cell16重训，movie2idx对齐")
print(f"{'Two-Tower v2':<22} {hr_tt2:>8.4f} {ndcg_tt2:>10.4f}  emb=32, tower=[64,32]")

hparams: {'emb_dim': 32, 'mlp_dims': [64, 32, 16], 'n_users': 6040, 'n_movies': 3706}
ncf_clean 加载成功

模型                        HR@10    NDCG@10  备注
----------------------------------------------------------
随机基准                     0.1000          —
NCF clean                0.4040     0.2076  Cell16重训，movie2idx对齐
Two-Tower v2             0.4100     0.2185  emb=32, tower=[64,32]
